# Comparator Table

This notebook reports reference models that help interpret the primary SRM composites.

**Comparator roles**

| Comparator | Nature | Input | Output | Why included |
|---|---|---|---|---|
| LDA | Linear Discriminant Analysis visit-separation direction | Imaging visit rows | Projection score | Tests whether visits can be separated by imaging patterns. |
| Regression reference | Ridge, PLS, or ElasticNet-style clinical target model | Imaging visit rows | Predicted clinical score | Tests whether imaging predicts clinical scales used as benchmarks. |

**Interpretation warning:** LDA Fisher-criterion separation and paired SRM-based progression `d_z` are related but not directly interchangeable.


In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd


def find_repo_root(start: Path) -> Path:
    for p in [start.resolve(), *start.resolve().parents]:
        if (p / "src").is_dir() and (p / "data").is_dir():
            return p
    raise FileNotFoundError("Could not find repo root")

REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.config import DEFAULT_CONFIG, set_global_seeds
from src.data.trackfa_pairs import infer_trackfa_feature_groups, trackfa_pairs_to_long
from src.eval.cv import interaction_loocv, lda_loocv, tune_and_run_regression_loocv
from src.eval.metrics import bootstrap_ci_d, clinical_change_effect_sizes, reference_effect_sizes
from src.eval.optimization import optimization_log, optimization_row, save_optimization_log
from src.features.selection import feature_stability_report
from src.models.srm_global import srm_global_loocv

set_global_seeds(DEFAULT_CONFIG.random_state)
pairs_path = REPO_ROOT / "data" / "processed" / "trackfa_pairs_drop3poms.csv"
if not pairs_path.exists():
    pairs_path = REPO_ROOT / "data" / "processed" / "trackfa_pairs.csv"
pairs_df = pd.read_csv(pairs_path)
long_df = trackfa_pairs_to_long(pairs_df)
groups = infer_trackfa_feature_groups(pairs_df)
imaging_cols = [c for c in groups.all_neuroimaging if c in long_df.columns]
subject_col = "pair_id"  # progression interval id, e.g. AAN001_V1V2
split_group_col = "subject"  # participant id; keeps V1V2 and V2V3 in the same fold
selection_method = "none"
selection_k = 8
LDA_CV_N_SPLITS = None  # Leave-One-Out cross-validation; LDA is fast enough for strict tuning.
REGRESSION_CV_N_SPLITS = DEFAULT_CONFIG.cv_n_splits
N_BOOT = 1000
RANDOM_SEED = DEFAULT_CONFIG.random_state
RIDGE_GRID = [1e-8, 1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1.0, 10.0]
COVARIANCE_SHRINKAGE_GRID = [0.0, 0.1, 0.25, 0.42, 0.45, 0.48, 0.5, 0.75, 1.0]
ELASTICNET_L1_RATIO_GRID = [0.2, 0.5, 0.8, 1.0]
PLS_COMPONENT_GRID = [1, 2, 3, 5, 8]
LDA_SHRINK_GRID = ["auto", *RIDGE_GRID]
REGRESSION_PARAM_SELECTION_METRIC = "d"
REGRESSION_MODEL_KINDS = ["ridge", "pls"]  # add "elasticnet" for the slower sparse CD backend
optimization_rows = []
print(f"Loaded {pairs_path.name}: {long_df.shape[0]} visit rows, {len(imaging_cols)} imaging features")
print({"selection_method": selection_method, "selection_k": selection_k, "lda_cv_n_splits": LDA_CV_N_SPLITS,
    "regression_cv_n_splits": REGRESSION_CV_N_SPLITS, "regression_param_selection_metric": REGRESSION_PARAM_SELECTION_METRIC})
def benchmark_table(model_name: str, d_score: float, ci_low: float, ci_high: float) -> pd.DataFrame:
    imaging_ref = reference_effect_sizes(
        long_df,
        imaging_cols=imaging_cols,
        scale_cols=(),
        subject_col=subject_col,
        visit_col="visit",
    )
    clinical_ref = clinical_change_effect_sizes(
        pairs_df,
        scale_cols=("FARS", "SARA"),
        pair_types=("V1V2", "V2V3"),
    )
    rows = [{"feature": model_name, "kind": "model", "d": d_score, "ci_low": ci_low, "ci_high": ci_high, "source_delta_col": np.nan, "pair_types": np.nan}]
    for scale in ("FARS", "SARA"):
        hit = clinical_ref[(clinical_ref["kind"] == "scale") & (clinical_ref["feature"] == scale)].head(1)
        if len(hit):
            r = hit.iloc[0].to_dict()
            rows.append({"feature": r["feature"], "kind": r["kind"], "d": r["d"], "ci_low": np.nan, "ci_high": np.nan, "source_delta_col": r.get("source_delta_col", np.nan), "pair_types": r.get("pair_types", np.nan)})
    top_img = imaging_ref[imaging_ref["kind"] == "imaging"].head(1)
    if len(top_img):
        r = top_img.iloc[0].to_dict()
        rows.append({"feature": r["feature"], "kind": r["kind"], "d": r["d"], "ci_low": np.nan, "ci_high": np.nan, "source_delta_col": r.get("source_delta_col", np.nan), "pair_types": r.get("pair_types", np.nan)})
    return pd.DataFrame(rows)

Loaded trackfa_pairs_drop3poms.csv: 414 visit rows, 146 imaging features
{'selection_method': 'none', 'selection_k': 8, 'lda_cv_n_splits': None, 'regression_cv_n_splits': 5, 'regression_param_selection_metric': 'd'}


## 1. LDA Visit-Separation Comparator

Linear Discriminant Analysis (LDA) finds a direction that separates visit labels. It is useful as a comparator, but the project target remains paired progression change on held-out participant groups.


In [2]:
import time

lda_trials = []
for covariance_shrinkage in COVARIANCE_SHRINKAGE_GRID:
    for shrink in LDA_SHRINK_GRID:
        start = time.time()
        res = lda_loocv(
            long_df,
            imaging_cols,
            subject_col=subject_col,
            visit_col="visit",
            selection_method=selection_method,
            k=selection_k,
            cv_n_splits=LDA_CV_N_SPLITS,
            random_seed=RANDOM_SEED,
        split_group_col=split_group_col,
            shrink=shrink,
            covariance_shrinkage=covariance_shrinkage,
            compute_ci=False,
        )
        row = optimization_row(
            model="LDA",
            params={
                "shrink": shrink,
                "covariance_shrinkage": covariance_shrinkage,
                "selection_method": selection_method,
                "regularization": "ridge_plus_covariance_shrinkage",
            },
            result=res,
            runtime_sec=time.time() - start,
            notes="outer held-out d_z grid; Fisher d and progression d_z are not interchangeable",
        )
        lda_trials.append((res, row))
        optimization_rows.append(row)

lda_optimization_df = optimization_log([row for _, row in lda_trials])
display(lda_optimization_df)
best_lda_row = lda_optimization_df.iloc[0]
lda_res = lda_loocv(
    long_df,
    imaging_cols,
    subject_col=subject_col,
    visit_col="visit",
    selection_method=selection_method,
    k=selection_k,
    cv_n_splits=LDA_CV_N_SPLITS,
    random_seed=RANDOM_SEED,
        split_group_col=split_group_col,
    shrink=best_lda_row["param_shrink"],
    covariance_shrinkage=float(best_lda_row["param_covariance_shrinkage"]),
    compute_ci=True,
)
pd.DataFrame([{
    "model": "LDA",
    "selection_method": selection_method,
    "best_shrink": lda_optimization_df.iloc[0].get("param_shrink", np.nan),
    "best_covariance_shrinkage": lda_optimization_df.iloc[0].get("param_covariance_shrinkage", np.nan),
    "d_z_from_scores": lda_res["d_score"],
    "ci_low": lda_res["d_ci_low"],
    "ci_high": lda_res["d_ci_high"],
    "n_subject_pairs": lda_res["n_subjects"],
}])


,model,feature_pool,objective,d_score,d_ci_low,d_ci_high,n_subjects,cv_mode,cv_n_splits,runtime_sec,notes,param_shrink,param_covariance_shrinkage,param_selection_method,param_regularization
0,LDA,all_imaging,d_score,0.414971,NaN,NaN,207,loo,207,1.102854,outer held-out d_z grid; Fisher d and progress...,10.0,1.0,none,ridge_plus_covariance_shrinkage
1,LDA,all_imaging,d_score,0.414956,NaN,NaN,207,loo,207,1.065444,outer held-out d_z grid; Fisher d and progress...,1.0,1.0,none,ridge_plus_covariance_shrinkage
2,LDA,all_imaging,d_score,0.414941,NaN,NaN,207,loo,207,1.120534,outer held-out d_z grid; Fisher d and progress...,0.1,1.0,none,ridge_plus_covariance_shrinkage
3,LDA,all_imaging,d_score,0.414938,NaN,NaN,207,loo,207,1.066392,outer held-out d_z grid; Fisher d and progress...,0.01,1.0,none,ridge_plus_covariance_shrinkage
4,LDA,all_imaging,d_score,0.414938,NaN,NaN,207,loo,207,1.013481,outer held-out d_z grid; Fisher d and progress...,0.001,1.0,none,ridge_plus_covariance_shrinkage
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85,LDA,all_imaging,d_score,-0.101072,NaN,NaN,207,loo,207,10.839593,outer held-out d_z grid; Fisher d and progress...,0.001,0.0,none,ridge_plus_covariance_shrinkage
86,LDA,all_imaging,d_score,-0.112473,NaN,NaN,207,loo,207,7.918365,outer held-out d_z grid; Fisher d and progress...,0.0001,0.0,none,ridge_plus_covariance_shrinkage
87,LDA,all_imaging,d_score,-0.115540,NaN,NaN,207,loo,207,8.406882,outer held-out d_z grid; Fisher d and progress...,0.00001,0.0,none,ridge_plus_covariance_shrinkage
88,LDA,all_imaging,d_score,-0.115921,NaN,NaN,207,loo,207,7.578818,outer held-out d_z grid; Fisher d and progress...,0.000001,0.0,none,ridge_plus_covariance_shrinkage


,model,selection_method,best_shrink,best_covariance_shrinkage,d_z_from_scores,ci_low,ci_high,n_subject_pairs
0,LDA,none,10.0,1.0,0.414971,0.269963,0.559454,207


## 2. Regression Reference

Regression models predict clinical targets from imaging features. These outputs are references, not the primary biomarker objective, because the project aims to measure imaging progression rather than optimise clinical-score prediction.


In [3]:
import time

regression_rows = []
regression_results = {}
for target in [c for c in ["FARS", "SARA"] if c in long_df.columns]:
    for model_kind in REGRESSION_MODEL_KINDS:
        start = time.time()
        res = tune_and_run_regression_loocv(
            long_df,
            imaging_cols,
            target_col=target,
            subject_col=subject_col,
            model_kind=model_kind,
            selection_method=selection_method,
            k=selection_k,
            visit_col="visit",
            cv_n_splits=REGRESSION_CV_N_SPLITS,
            param_selection_metric=REGRESSION_PARAM_SELECTION_METRIC,
        )
        regression_results[(target, model_kind)] = res
        params = {"target": target, "model_kind": model_kind, "selection_method": selection_method, "param_selection_metric": REGRESSION_PARAM_SELECTION_METRIC}
        optimization_rows.append(optimization_row(
            model=f"Regression {model_kind}",
            params=params,
            result=res,
            runtime_sec=time.time() - start,
            notes="inner hyperparameters selected by paired d_z of OOF predictions",
        ))
        regression_rows.append({
            "target": target,
            "model": model_kind,
            "selection_method": selection_method,
            "param_selection_metric": REGRESSION_PARAM_SELECTION_METRIC,
            "rmse": res["rmse"],
            "r2": res["r2"],
            "d_z_from_oof_predictions": res["d_score"],
            "ci_low": res["d_ci_low"],
            "ci_high": res["d_ci_high"],
            "n_subject_pairs": res["n_subjects"],
        })
regression_df = pd.DataFrame(regression_rows)
optimization_df = optimization_log(optimization_rows)
log_path = save_optimization_log(optimization_df, REPO_ROOT / "results" / "comparator_optimization_log.csv")
print("Saved optimization log:", log_path)
display(regression_df.sort_values("d_z_from_oof_predictions", ascending=False))
display(optimization_df)


Saved optimization log: /Users/robertwang/Documents/New_project/biomarkers/results/comparator_optimization_log.csv


,target,model,selection_method,param_selection_metric,rmse,r2,d_z_from_oof_predictions,ci_low,ci_high,n_subject_pairs
3,SARA,pls,none,d,5.685991,0.354164,0.445244,0.311427,0.596963,207
1,FARS,pls,none,d,11.574195,0.289426,0.425829,0.296026,0.577601,207
2,SARA,ridge,none,d,16.067757,-4.157275,0.382287,0.249400,0.530987,207
0,FARS,ridge,none,d,44.543737,-9.524489,0.347204,0.213636,0.496834,207


,model,feature_pool,objective,d_score,d_ci_low,d_ci_high,n_subjects,cv_mode,cv_n_splits,runtime_sec,notes,param_shrink,param_covariance_shrinkage,param_selection_method,param_regularization,param_target,param_model_kind,param_param_selection_metric
0,Regression pls,all_imaging,d_score,0.445244,0.311427,0.596963,207,group_kfold,5,0.161090,inner hyperparameters selected by paired d_z o...,NaN,NaN,none,NaN,SARA,pls,d
1,Regression pls,all_imaging,d_score,0.425829,0.296026,0.577601,207,group_kfold,5,0.147292,inner hyperparameters selected by paired d_z o...,NaN,NaN,none,NaN,FARS,pls,d
2,LDA,all_imaging,d_score,0.414971,NaN,NaN,207,loo,207,1.102854,outer held-out d_z grid; Fisher d and progress...,10.0,1.0,none,ridge_plus_covariance_shrinkage,NaN,NaN,NaN
3,LDA,all_imaging,d_score,0.414956,NaN,NaN,207,loo,207,1.065444,outer held-out d_z grid; Fisher d and progress...,1.0,1.0,none,ridge_plus_covariance_shrinkage,NaN,NaN,NaN
4,LDA,all_imaging,d_score,0.414941,NaN,NaN,207,loo,207,1.120534,outer held-out d_z grid; Fisher d and progress...,0.1,1.0,none,ridge_plus_covariance_shrinkage,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
89,LDA,all_imaging,d_score,-0.101072,NaN,NaN,207,loo,207,10.839593,outer held-out d_z grid; Fisher d and progress...,0.001,0.0,none,ridge_plus_covariance_shrinkage,NaN,NaN,NaN
90,LDA,all_imaging,d_score,-0.112473,NaN,NaN,207,loo,207,7.918365,outer held-out d_z grid; Fisher d and progress...,0.0001,0.0,none,ridge_plus_covariance_shrinkage,NaN,NaN,NaN
91,LDA,all_imaging,d_score,-0.115540,NaN,NaN,207,loo,207,8.406882,outer held-out d_z grid; Fisher d and progress...,0.00001,0.0,none,ridge_plus_covariance_shrinkage,NaN,NaN,NaN
92,LDA,all_imaging,d_score,-0.115921,NaN,NaN,207,loo,207,7.578818,outer held-out d_z grid; Fisher d and progress...,0.000001,0.0,none,ridge_plus_covariance_shrinkage,NaN,NaN,NaN


## 3. Clinical Benchmark Table

The final display compares the best comparator row with FARS, SARA, and the top single imaging feature using the same paired Cohen's `d_z` convention where applicable.


In [4]:
best_reg = regression_df.sort_values("d_z_from_oof_predictions", ascending=False).head(1)
if len(best_reg):
    row = best_reg.iloc[0]
    model_name = f"Regression {row['model']} ({row['target']})"
    model_d = row["d_z_from_oof_predictions"]
    model_lo = row["ci_low"]
    model_hi = row["ci_high"]
else:
    model_name, model_d, model_lo, model_hi = "Regression reference", np.nan, np.nan, np.nan
lda_table = pd.DataFrame([{"feature": "LDA", "kind": "model", "d": lda_res["d_score"], "ci_low": lda_res["d_ci_low"], "ci_high": lda_res["d_ci_high"]}])
reg_table = benchmark_table(model_name, model_d, model_lo, model_hi)
display(pd.concat([lda_table, reg_table], ignore_index=True))

,feature,kind,d,ci_low,ci_high,source_delta_col,pair_types
0,LDA,model,0.414971,0.269963,0.559454,NaN,NaN
1,Regression pls (SARA),model,0.445244,0.311427,0.596963,NaN,NaN
2,FARS,scale,0.407427,NaN,NaN,delta_mfars_total,"V1V2,V2V3"
3,SARA,scale,0.405463,NaN,NaN,delta_sara_total,"V1V2,V2V3"
4,cerebellumFS,imaging,-0.667820,NaN,NaN,NaN,NaN
